# Arbol de decisiones

Resolver el problema de supervivencia del Titanic con un arbol de decisiones usando el dataset preprocesado.

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Carga y limpieza

In [20]:
df = pd.read_csv('dataset.csv')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)

df_model.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,2,0,True,False,True
1,1,1,38.0,1,0,71.2833,2,0,False,False,False
2,1,3,26.0,0,0,7.9250,1,1,False,False,True
3,1,1,35.0,1,0,53.1000,2,0,False,False,True
4,0,3,35.0,0,0,8.0500,1,1,True,False,True


## 2. Separar variables y dividir

In [21]:
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 3. Entrenamiento y evaluacion

In [22]:
model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=2,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClasification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7640449438202247

Clasification report:
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       110
           1       0.75      0.57      0.65        68

    accuracy                           0.76       178
   macro avg       0.76      0.73      0.74       178
weighted avg       0.76      0.76      0.76       178

Confusion matrix:
[[97 13]
 [29 39]]


## 4. Importancia de variables

In [23]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

Sex_male      0.548202
Pclass        0.165012
Age           0.111449
Fare          0.094181
FamilySize    0.060043
Embarked_Q    0.021113
SibSp         0.000000
Parch         0.000000
IsAlone       0.000000
Embarked_S    0.000000
dtype: float64

## 5. Busqueda simple de hiperparametros

In [24]:
best_score = 0.0
best_depth = None

for depth in range(2, 11):
    temp_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    temp_model.fit(X_train, y_train)
    score = temp_model.score(X_test, y_test)
    print(f'max_depth={depth} -> accuracy={score:.4f}')
    if score > best_score:
        best_score = score
        best_depth = depth

print('\nMejor profundidad:', best_depth)
print('Mejor accuracy:', best_score)

max_depth=2 -> accuracy=0.7640
max_depth=3 -> accuracy=0.8090
max_depth=4 -> accuracy=0.8034
max_depth=5 -> accuracy=0.7640
max_depth=6 -> accuracy=0.7584
max_depth=7 -> accuracy=0.7416
max_depth=8 -> accuracy=0.7697
max_depth=9 -> accuracy=0.7865
max_depth=10 -> accuracy=0.7697

Mejor profundidad: 3
Mejor accuracy: 0.8089887640449438


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

top = importances.sort_values(ascending=False).head(4)
top_txt = ", ".join([f"{idx} ({val:.3f})" for idx, val in top.items()])

print("Conclusion y explicacion (resumen automatico)")
print(f"- Accuracy: {acc:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor profundidad: {best_depth} con accuracy {best_score:.3f}")
print(f"- Variables mas importantes: {top_txt}")

Conclusion y explicacion (resumen automatico)
- Accuracy: 0.764
- Clase 0 (no sobrevive): precision=0.770, recall=0.882
- Clase 1 (sobrevive): precision=0.750, recall=0.574
- Matriz de confusion: TN=97, FP=13, FN=29, TP=39 -> sesgo conservador
- Mejor profundidad: 3 con accuracy 0.809
- Variables mas importantes: Sex_male (0.548), Pclass (0.165), Age (0.111), Fare (0.094)


## 6. Conclusion y explicacion de resultados (con datos reales)

- **Rendimiento**: accuracy = 0.764. El modelo acierta aproximadamente el 76.4% de los casos en test.
- **Clase 0 (no sobrevive)**: precision = 0.770, recall = 0.882. Predice bien la clase negativa y recupera la mayoria de no sobrevivientes.
- **Clase 1 (sobrevive)**: precision = 0.750, recall = 0.574. Se pierden varios sobrevivientes, lo que baja el recall de la clase positiva.
- **Matriz de confusion**: TN=97, FP=13, FN=29, TP=39. El sesgo es **conservador** (mas falsos negativos que falsos positivos).
- **Profundidad optima**: mejor `max_depth` = 3 con accuracy = 0.809 en el barrido simple.
- **Variables mas influyentes**: `Sex_male` (0.548), `Pclass` (0.165), `Age` (0.111), `Fare` (0.094). Estas variables dominan la decision del arbol.
- **Conclusión general**: el arbol es interpretable y funciona razonablemente bien, pero tiende a subestimar la clase de sobrevivientes. Ajustar hiperparametros o balancear clases podria mejorar el recall de la clase 1.